# Landing → Bronze: leituras

**Fonte:** 5 arquivos CSV particionados por ano (`leituras_2022.csv` … `leituras_2026.csv`), separador vírgula, com cabeçalho.
**Destino:** `venergy.bronze.leituras` (Delta).

**Decisões desta camada**
- Conteúdo preservado: todas as colunas lidas como texto (schema explícito, todas `StringType`), sem limpeza, deduplicação ou conversão. O tratamento é feito na Silver.
- Colunas de rastreio: `source_file` (arquivo de origem) e `ingestion_datetime` (momento da carga), padrão do time.
- Carga completa com `overwrite`: o dataset é uma extração única, então reexecutar o notebook não duplica registros (o Workflow precisa ser reexecutável de ponta a ponta).
- Validação antes de gravar: o cabeçalho de **cada** arquivo é conferido com o schema esperado (`enforceSchema = false`), uma linha malformada derruba a carga (`FAILFAST`) e a contagem por arquivo é validada **antes** do `overwrite`. Uma carga inválida nunca substitui a Bronze.
- Observação: o leitor CSV do Spark transforma campo vazio em `null`. Os demais marcadores de ausência (`N/A`, `NULL`) chegam como texto, e a sentinela `-1` chega como um número válido (passa em `try_cast`). Todos são documentados e tratados na Silver.

As falhas encontradas estão em `docs/auditoria_dados/auditoria_leituras.md`.

In [ ]:
catalog = "venergy"
landing_schema = "landing"
bronze_schema = "bronze"
landing_volume = "landing_volume"
volume_path = f"/Volumes/{catalog}/{landing_schema}/{landing_volume}"

leituras_path = f"{volume_path}/dados/leituras"
bronze_table = f"{catalog}.{bronze_schema}.leituras"

EXPECTED_COLUMNS = [
    "id_leitura", "id_uc", "id_medidor", "data_leitura", "mes_referencia",
    "leitura_anterior", "leitura_atual", "consumo_kwh", "tipo_leitura",
]
EXPECTED_ROWS_PER_FILE = {
    "leituras_2022.csv": 271_744,
    "leituras_2023.csv": 544_546,
    "leituras_2024.csv": 545_652,
    "leituras_2025.csv": 546_908,
    "leituras_2026.csv": 273_618,
}

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{landing_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{landing_schema}.{landing_volume}")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

schema_bruto = StructType([StructField(c, StringType(), True) for c in EXPECTED_COLUMNS])

df_leituras = (
    spark.read
    .schema(schema_bruto)
    .option("header", True)
    .option("enforceSchema", False)  # valida o cabeçalho de CADA arquivo contra EXPECTED_COLUMNS
    .option("mode", "FAILFAST")      # linha com campos a mais ou a menos derruba a carga
    .option("sep", ",")
    .option("encoding", "UTF-8")
    .csv(f"{leituras_path}/leituras_*.csv")
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("ingestion_datetime", F.current_timestamp())
)

In [ ]:
# Validação ANTES de gravar: uma carga inválida não chega a substituir a Bronze.
contagem = {r["source_file"]: r["count"] for r in df_leituras.groupBy("source_file").count().collect()}
print(contagem)
assert contagem == EXPECTED_ROWS_PER_FILE, f"Contagem por arquivo diverge do bruto: {contagem}"

In [ ]:
(
    df_leituras.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

In [ ]:
total_bronze = spark.table(bronze_table).count()
assert total_bronze == 2_182_468, f"Bronze com {total_bronze} linhas"
print("OK: 2.182.468 linhas carregadas em", bronze_table)

## Evidências das falhas (consultas sobre a Bronze)
As consultas abaixo reproduzem os principais achados do relatório de auditoria. Elas não alteram dados.

In [ ]:
t = bronze_table

# 1. Formatos de data (troca cada dígito por 9 e conta os padrões)
display(spark.sql(f"""
  SELECT regexp_replace(data_leitura, '[0-9]', '9') AS formato, COUNT(*) AS linhas
  FROM {t} GROUP BY 1 ORDER BY 2 DESC"""))

# 2. Duplicatas exatas e leituras clonadas (id LTX…)
display(spark.sql(f"""
  SELECT
    (SELECT COUNT(*) FROM {t}) - (SELECT COUNT(*) FROM (SELECT DISTINCT {", ".join(EXPECTED_COLUMNS)} FROM {t})) AS duplicatas_exatas,
    (SELECT COUNT(*) FROM {t} WHERE id_leitura LIKE 'LTX%') AS leituras_ltx"""))

# 3. Consumo negativo por causa provável
display(spark.sql(f"""
  SELECT CASE
           WHEN try_cast(leitura_anterior AS DOUBLE) > try_cast(leitura_atual AS DOUBLE) THEN 'troca de medidor'
           WHEN try_cast(leitura_anterior AS DOUBLE) IS NULL OR try_cast(leitura_anterior AS DOUBLE) < 0 THEN 'sem leitura anterior'
           WHEN try_cast(consumo_kwh AS DOUBLE) = -(try_cast(leitura_atual AS DOUBLE) - try_cast(leitura_anterior AS DOUBLE)) THEN 'sinal invertido'
           ELSE 'outro'
         END AS causa, COUNT(*) AS linhas
  FROM {t} WHERE try_cast(consumo_kwh AS DOUBLE) < 0 GROUP BY 1 ORDER BY 2 DESC"""))

# 4. Marcadores de ausência em leitura_anterior
display(spark.sql(f"""
  SELECT coalesce(leitura_anterior, '<null>') AS valor, COUNT(*) AS linhas
  FROM {t} WHERE leitura_anterior IS NULL OR leitura_anterior IN ('N/A', 'NULL', '-1')
  GROUP BY 1 ORDER BY 2 DESC"""))